# Exploratory Data Analysis

## Objectives

This notebook will use the processed retail datasets to:

- calculate the principal business performance metrics;
- describe invoice-value distributions;
- analyse sales trends over time;
- identify high-performing products;
- compare geographic markets;
- investigate cancellations and adjustments;
- produce interactive visualisations for the Streamlit dashboard.

Completed sales will be used for revenue analysis. Returns and adjustments will be analysed separately.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
project_root = Path.cwd()

if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

processed_data_dir = project_root / "data" / "processed"

completed_sales_path = (
    processed_data_dir / "completed_sales.parquet"
)

returns_adjustments_path = (
    processed_data_dir / "returns_adjustments.parquet"
)

assert completed_sales_path.exists()
assert returns_adjustments_path.exists()

print(f"Project root: {project_root}")
print(f"Processed data: {processed_data_dir}")

Project root: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project2-Online Retail Transaction Analysis/CI-DA-Project-2-Online-Retail-Transaction-Analysis
Processed data: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project2-Online Retail Transaction Analysis/CI-DA-Project-2-Online-Retail-Transaction-Analysis/data/processed


In [3]:
sales_df = pd.read_parquet(completed_sales_path)
returns_df = pd.read_parquet(returns_adjustments_path)

assert len(sales_df) == 524_878
assert len(returns_df) == 11_763
assert sales_df["LineRevenue"].gt(0).all()
assert sales_df["TransactionType"].eq("Completed sale").all()

print(f"Completed sales rows: {len(sales_df):,}")
print(f"Returns and adjustment rows: {len(returns_df):,}")

Completed sales rows: 524,878
Returns and adjustment rows: 11,763


## Invoice-level analysis

The processed sales dataset contains one row per product line. Average transaction value must therefore be calculated after aggregating product lines to invoice level. Calculating the mean directly from `LineRevenue` would give average line value rather than average completed transaction value.

In [4]:
invoice_summary = (
    sales_df.groupby("InvoiceNo", as_index=False)
    .agg(
        InvoiceRevenue=("LineRevenue", "sum"),
        InvoiceDate=("InvoiceDate", "min"),
        CustomerID=("CustomerID", "first"),
        Country=("Country", "first"),
        Units=("Quantity", "sum"),
        ProductLines=("StockCode", "count"),
    )
)

assert len(invoice_summary) == sales_df["InvoiceNo"].nunique()

invoice_summary.head()

,InvoiceNo,InvoiceRevenue,InvoiceDate,CustomerID,Country,Units,ProductLines
0,536365,139.12,2010-12-01 08:26:00,17850,United Kingdom,40,7
1,536366,22.20,2010-12-01 08:28:00,17850,United Kingdom,12,2
2,536367,278.73,2010-12-01 08:34:00,13047,United Kingdom,83,12
3,536368,70.05,2010-12-01 08:34:00,13047,United Kingdom,15,4
4,536369,17.85,2010-12-01 08:35:00,13047,United Kingdom,3,1


In [5]:
total_revenue = sales_df["LineRevenue"].sum()
completed_invoices = invoice_summary["InvoiceNo"].nunique()
average_invoice_value = invoice_summary["InvoiceRevenue"].mean()
median_invoice_value = invoice_summary["InvoiceRevenue"].median()
unique_customers = sales_df["CustomerID"].nunique()
unique_products = sales_df["StockCode"].nunique()
units_sold = sales_df["Quantity"].sum()
countries_served = sales_df["Country"].nunique()

kpi_summary = pd.DataFrame(
    {
        "Metric": [
            "Total completed sales revenue",
            "Completed invoices",
            "Average invoice value",
            "Median invoice value",
            "Unique customer identifiers",
            "Unique products",
            "Units sold",
            "Countries served",
        ],
        "Value": [
            f"£{total_revenue:,.2f}",
            f"{completed_invoices:,}",
            f"£{average_invoice_value:,.2f}",
            f"£{median_invoice_value:,.2f}",
            f"{unique_customers:,}",
            f"{unique_products:,}",
            f"{units_sold:,}",
            f"{countries_served:,}",
        ],
    }
)

kpi_summary

,Metric,Value
0,Total completed sales revenue,"£10,642,110.80"
1,Completed invoices,"19,960"
2,Average invoice value,£533.17
3,Median invoice value,£303.30
4,Unique customer identifiers,"4,338"
5,Unique products,"3,922"
6,Units sold,"5,572,420"
7,Countries served,38


### KPI interpretation

The completed-sales dataset generated approximately £10.64 million across 19,960 completed invoices.

The mean invoice value is approximately £533.17, while the median is £303.30. The substantially higher mean suggests that a relatively small number of high-value invoices pull the average upward. The median therefore represents a more typical invoice, while the mean remains useful for overall revenue planning.

The customer count includes identifier `15287` because general sales metrics retain its valid transaction values. It will not be used in customer
segmentation.

In [6]:
invoice_value_statistics = (
    invoice_summary["InvoiceRevenue"]
    .describe(
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

invoice_value_statistics

count    19,960.00
mean        533.17
std       1,780.41
min           0.38
1%            4.68
25%         151.70
50%         303.30
75%         493.46
95%       1,585.92
99%       4,821.17
max     168,469.60
Name: InvoiceRevenue, dtype: float64

In [7]:
invoice_value_99th_percentile = (
    invoice_summary["InvoiceRevenue"].quantile(0.99)
)

invoice_values_for_chart = invoice_summary.loc[
    invoice_summary["InvoiceRevenue"]
    <= invoice_value_99th_percentile
].copy()

fig = px.histogram(
    invoice_values_for_chart,
    x="InvoiceRevenue",
    nbins=60,
    title=(
        "Distribution of Completed Invoice Values "
        "(up to the 99th Percentile)"
    ),
    labels={
        "InvoiceRevenue": "Invoice value (£)",
        "count": "Number of invoices",
    },
)

fig.update_layout(
    bargap=0.05,
    showlegend=False,
)

fig.show()

The invoice-value distribution is strongly right-skewed. Most invoices are concentrated at lower values, while a small number of high-value invoices extend the upper tail.

The chart is limited to the 99th percentile for readability. The underlying data has not been removed or changed, and all invoices remain included in the KPI calculations.